<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/Python-2026/Practice/FastAPI_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Вариант 1. FastAPI + ML (85 задач: setup + 80 основных)

> **Тема:** Библиотека — веб-сервис, который принимает данные о книге/читателе и возвращает предсказание ML-модели (вернёт ли читатель книгу вовремя). Плюс API для работы с каталогом и историей предсказаний.
> **Цель:** покрыть **всю практическую теорию** FastAPI для Data Science: маршрутизация → Pydantic → интеграция ML → обработка ошибок → зависимости → БД → асинхронность → тесты → безопасность → деплой (без повторения основ Docker).
> **Стек:** `fastapi`, `uvicorn`, `pydantic` (v2), `scikit-learn`, `joblib`, `pytest`, `httpx`, `SQLAlchemy` (SQLite).
> **Предусловие:** у вас есть обученная модель `model.joblib` из курса Scikit-learn.

---

## 🟢 Уровень 0. Setup: первый сервер (0.1–0.5)

**Что изучается:** установка FastAPI и Uvicorn, структура проекта, первый `@app.get`, Swagger UI (`/docs`), `--reload`, разница между `uvicorn main:app` и `uvicorn main:app --reload`.

**Задача 0.1 (установка и первый эндпоинт).** Создайте `main.py` с `FastAPI()` и эндпоинтом `GET /`, возвращающим `{"status": "ok", "service": "library-ml"}`. Запустите: `uvicorn main:app --reload`. Ожидаемый вывод: `http://127.0.0.1:8000/` возвращает JSON. В комментариях поясните: что делает `uvicorn` (ASGI-сервер, который запускает FastAPI); что такое ASGI и чем отличается от WSGI (async vs sync); зачем `--reload` (автоперезапуск при изменении кода — только для dev); почему `main:app` (файл `main.py`, объект `app`). Подсказка: `from fastapi import FastAPI; app = FastAPI()`.

**Задача 0.2 (Swagger UI: `/docs` и `/redoc`).** Откройте `http://127.0.0.1:8000/docs` и `/redoc`. Ожидаемый вывод: автоматическая интерактивная документация. В комментариях поясните: почему FastAPI генерирует доки автоматически (на основе Pydantic и type hints); чем `/docs` (Swagger) отличается от `/redoc`; как попробовать эндпоинт прямо в браузере (Try it out); зачем это нужно (тестирование без Postman). Подсказка: просто откройте `/docs` в браузере.

**Задача 0.3 (структура проекта).** Создайте структуру:
```
app/
├── __init__.py
├── main.py
├── schemas.py   # Pydantic-модели
├── models.py    # ML-модель
├── routers/
│   ├── __init__.py
│   └── predict.py
└── core/
    └── config.py
```
Перенесите эндпоинт в `app/routers/predict.py` (используя `APIRouter`), подключите в `main.py` через `app.include_router()`. Ожидаемый вывод: `/` работает, код разбит по файлам. В комментариях поясните: зачем `APIRouter` (разделение больших приложений); почему `schemas.py` отдельно от `models.py` (Pydantic-валидация vs ML-логика); как `include_router` собирает всё в одно приложение. Подсказка: `from fastapi import APIRouter; router = APIRouter(prefix="/predict", tags=["predict"])`.

**Задача 0.4 (проверка версий и окружения).** Проверьте: `pip show fastapi pydantic uvicorn scikit-learn joblib`. Ожидаемый вывод: `fastapi 0.109+`, `pydantic 2.x`, `uvicorn 0.27+`. В комментариях поясните: почему Pydantic v2 критичен (полностью переписан, `BaseModel.model_dump` вместо `dict`); что будет, если Pydantic v1 (многие туториалы устарели); как проверить совместимость (`pip check`). Подсказка: `pydantic.VERSION` в Python-консоли.

**Задача 0.5 (`config.py` + `.env`).** Создайте `app/core/config.py` через `pydantic-settings` (`BaseSettings`), который читает `MODEL_PATH`, `DB_URL`, `API_KEY` из `.env`. Ожидаемый вывод: `settings.MODEL_PATH == "model.joblib"`. В комментариях поясните: зачем `BaseSettings` (читает переменные окружения и `.env` автоматически); почему **нельзя** хардкодить пути и ключи; как `pydantic-settings` валидирует типы из env; почему `.env` в `.gitignore`. Подсказка: `pip install pydantic-settings`; `from pydantic_settings import BaseSettings; class Settings(BaseSettings): model_path: str = "model.joblib"`.

---

## 🔵 Уровень 1. Маршрутизация и параметры (1–8)

**Что изучается:** `@app.get/post/put/delete`, path parameters, query parameters, `Path`/`Query` валидаторы, `status_code`, `response_model`, `tags`.

**Задача 1 (path parameters).** Создайте `GET /books/{book_id}` — возвращает `{"book_id": book_id, "title": f"Book {book_id}"}`. Ожидаемый вывод: `GET /books/42` → `{"book_id": 42, "title": "Book 42"}`. В комментариях поясните: что такое path parameter (часть URL); как FastAPI валидирует тип (`book_id: int`); что будет при `GET /books/abc` (422 Unprocessable Entity — авто-валидация); почему это лучше ручной проверки. Подсказка: `@app.get("/books/{book_id}") def get_book(book_id: int): ...`.

**Задача 2 (query parameters).** Добавьте `GET /books?genre=роман&year_min=2000&limit=10` с параметрами `genre: str | None = None`, `year_min: int = 1500`, `limit: int = 10`. Ожидаемый вывод: все три параметра опциональны, значения по умолчанию работают. В комментариях поясните: чем query params отличаются от path (query — фильтры, path — идентификатор); как FastAPI выводит типы из аннотаций; почему `None` (optional) — стандарт для фильтров; как `limit` ограничить (`Query(10, le=100)`). Подсказка: `@app.get("/books") def list_books(genre: str | None = None, limit: int = 10): ...`.

**Задача 3 (`Path` и `Query` с валидацией).** Используйте `Path(gt=0)` для `book_id` и `Query(..., le=100)` для `limit`. Ожидаемый вывод: `GET /books/-1` → 422; `GET /books?limit=200` → 422. В комментариях поясните: зачем `Path`/`Query` объекты (доп. валидация: `gt`, `ge`, `lt`, `le`, `regex`, `min_length`); почему валидация на уровне схемы лучше ручной; как ошибка выглядит в JSON (поле `detail`). Подсказка: `from fastapi import Path, Query; book_id: int = Path(gt=0)`.

**Задача 4 (POST + body).** Создайте `POST /predict` — принимает JSON `{"days_held": 14, "genre": "роман"}` и возвращает `{"prediction": 1}`. Ожидаемый вывод: POST с JSON работает; без тела — 422. В комментариях поясните: чем POST отличается от GET (тело запроса, не идемпотентен); как FastAPI читает JSON (автоматически парсит в Pydantic-модель, см. уровень 2); почему GET **не** должен иметь тело. Подсказка: `from fastapi import FastAPI; @app.post("/predict") def predict(data: dict): ...`.

**Задача 5 (`status_code`).** Установите `status_code=201` для `POST /books` (создание) и `204` для `DELETE /books/{id}`. Ожидаемый вывод: Swagger показывает коды 201/204. В комментариях поясните: зачем правильные status codes (REST-семантика); что значит 200, 201, 204, 400, 404, 422, 500; почему 422 — стандарт FastAPI для ошибок валидации. Подсказка: `@app.post("/books", status_code=201)`.

**Задача 6 (`response_model`).** Создайте Pydantic-модель `BookResponse` и установите `@app.get("/books/{id}", response_model=BookResponse)`. Ожидаемый вывод: Swagger показывает схему ответа; лишние поля из внутреннего объекта **отфильтровываются**. В комментариях поясните: зачем `response_model` (фильтрация полей, документация, безопасность); почему он защищает от утечки паролей/токенов; чем отличается от `-> BookResponse` в аннотации (эквивалентно, но `response_model` явнее). Подсказка: `@app.get("/books/{id}", response_model=BookResponse)`.

**Задача 7 (`tags` + `summary` + `description`).** Добавьте в роутер `tags=["books"]`, а в эндпоинт — `summary="Получить книгу"`, `description="Возвращает книгу по ID."`. Ожидаемый вывод: Swagger группирует эндпоинты по тегам. В комментариях поясните: зачем `tags` (группировка в Swagger); зачем `summary`/`description` (человекочитаемая дока); как это влияет на `openapi.json`. Подсказка: `@app.get("/books/{id}", tags=["books"], summary="...")`.

**Задача 8 (`APIRouter` с `prefix`).** Создайте `books_router = APIRouter(prefix="/books", tags=["books"])` и подключите в `main.py`. Ожидаемый вывод: все эндпоинты `/books/...` работают. В комментариях поясните: зачем `prefix` (не дублировать `/books` в каждом декораторе); почему `tags` на роутере (все эндпоинты группы); как `include_router(books_router)` собирает приложение. Подсказка: `app.include_router(books_router)`.

---

## 🟣 Уровень 2. Pydantic v2: схемы и валидация (9–18)

**Что изучается:** `BaseModel`, `Field`, `field_validator`, `model_validator`, `ConfigDict`, `model_dump`/`model_dump_json`, вложенные модели, `Literal`, `Enum`, `EmailStr`, response-модели.

**Задача 9 (`BaseModel`: база).** Создайте `PredictRequest(BaseModel)` с `days_held: int`, `genre: str`. Ожидаемый вывод: FastAPI автоматически валидирует и документирует. В комментариях поясните: что такое Pydantic-модель (схема данных + валидация); почему **все** тела запросов должны быть Pydantic-моделями (не `dict`); что будет при неверном типе (`days_held: "abc"` → 422). Подсказка: `from pydantic import BaseModel; class PredictRequest(BaseModel): days_held: int; genre: str`.

**Задача 10 (`Field` с валидацией).** Добавьте `Field(..., gt=0, le=365, description="Дней на руках")` для `days_held`. Ожидаемый вывод: `days_held=-5` → 422; в Swagger видно описание и ограничения. В комментариях поясните: зачем `Field` (дефолты, ограничения, описание); какие параметры (`gt`, `ge`, `lt`, `le`, `min_length`, `max_length`, `pattern`); что делает `...` (обязательное поле). Подсказка: `days_held: int = Field(..., gt=0, le=365)`.

**Задача 11 (`Literal` и `Enum`).** Ограничьте `genre` значениями: `Literal["роман", "детектив", "фантастика", "поэма"]` (или `Enum`). Ожидаемый вывод: `genre="комикс"` → 422; Swagger показывает выпадающий список. В комментариях поясните: чем `Literal` отличается от `Enum` (первый — проще, второй — переиспользуемый, с методами); почему это критично для ML (модель обучена на конкретных категориях); что будет при unseen category (см. задачу 22). Подсказка: `from typing import Literal; genre: Literal["роман", "детектив", "фантастика", "поэма"]`.

**Задача 12 (`field_validator`).** Напишите валидатор: если `genre == "поэма"`, то `days_held` не может быть > 100. Ожидаемый вывод: `genre="поэма", days_held=150` → 422 с понятным сообщением. В комментариях поясните: как работает `@field_validator` (кастомная логика); чем `mode="before"` отличается от `mode="after"` (до/после приведения типов); как вернуть ошибку (`raise ValueError`); почему бизнес-правила лучше держать в Pydantic. Подсказка: `from pydantic import field_validator; @field_validator("days_held") @classmethod def check_poem(cls, v, info): ...`.

**Задача 13 (`model_validator`).** Напишите `@model_validator(mode="after")`: если `days_held > 30` и `genre == "детектив"`, поднять ошибку «детективы редко держат так долго». Ожидаемый вывод: комбинация полей валидируется. В комментариях поясните: чем `model_validator` отличается от `field_validator` (доступ ко **всем** полям); когда нужен (кросс-валидация); почему `mode="after"` — стандарт (после приведения типов). Подсказка: `@model_validator(mode="after") def check_consistency(self): ...`.

**Задача 14 (вложенные модели).** Создайте `BatchPredictRequest` с `items: list[PredictRequest]` (до 100 штук). Ожидаемый вывод: `POST /predict/batch` принимает массив и возвращает массив предсказаний. В комментариях поясните: зачем батчинг (быстрее в 10–100 раз, чем по одному); как ограничить длину (`Field(..., max_length=100)`); почему батчинг критичен для ML (векторизация). Подсказка: `class BatchPredictRequest(BaseModel): items: list[PredictRequest] = Field(..., max_length=100)`.

**Задача 15 (`response_model` + `ConfigDict`).** Создайте `PredictResponse` с `prediction: int`, `probability: float`, `model_version: str`. Настройте `model_config = ConfigDict(from_attributes=True)` (для ORM-совместимости). Ожидаемый вывод: ответ строго по схеме. В комментариях поясните: зачем `from_attributes` (позволяет создавать Pydantic-модель из ORM-объекта, задача 40); почему `response_model` фильтрует лишние поля; как `model_dump()` конвертирует в `dict`. Подсказка: `from pydantic import ConfigDict; model_config = ConfigDict(from_attributes=True)`.

**Задача 16 (`model_dump` / `model_dump_json`).** Продемонстрируйте: `req.model_dump()` (dict), `req.model_dump_json()` (строка JSON), `req.model_dump(exclude={"genre"})`. Ожидаемый вывод: три разных представления. В комментариях поясните: зачем `model_dump` (конвертация для pandas/ML); зачем `exclude`/`include` (убрать лишнее); чем Pydantic v2 отличается от v1 (`.dict()` → `.model_dump()` — старое удалено). Подсказка: `req.model_dump()`.

**Задача 17 (`EmailStr` + `HttpUrl`).** В `ReaderCreate` используйте `EmailStr` для `email` и `HttpUrl` для `avatar_url`. Ожидаемый вывод: невалидный email → 422. В комментариях поясните: зачем специальные типы (`EmailStr` валидирует формат); что нужно установить (`pip install "pydantic[email]"`); почему ручная проверка regex — плохая идея. Подсказка: `from pydantic import EmailStr; email: EmailStr`.

**Задача 18 (`Field` в ответе + `json_schema_extra`).** Добавьте пример в `PredictRequest` через `model_config = ConfigDict(json_schema_extra={"example": {"days_held": 14, "genre": "роман"}})`. Ожидаемый вывод: Swagger показывает готовый пример — можно нажать «Try it out». В комментариях поясните: зачем примеры в Swagger (UX для frontend/тестировщиков); как `json_schema_extra` улучшает документацию; почему это лучше, чем объяснять в README. Подсказка: `model_config = ConfigDict(json_schema_extra={"example": {...}})`.

---

## 🟠 Уровень 3. Интеграция ML-модели (19–28)

**Что изучается:** загрузка `joblib` при старте приложения (`lifespan`), инференс, `predict` vs `predict_proba`, конвертация Pydantic → pandas → модель, batch-инференс, логирование предсказаний.

**Задача 19 (`lifespan`: загрузка модели при старте).** Загрузите `model.joblib` один раз при старте приложения через `lifespan` (не при каждом запросе). Ожидаемый вывод: модель загружается в память 1 раз; `/predict` работает быстро. В комментариях поясните: почему **нельзя** загружать модель внутри `def predict()` (каждый запрос — 1–2 секунды на загрузку); что такое `lifespan` (современная замена `@app.on_event("startup")`); как хранить модель в `app.state`. Подсказка: `from contextlib import asynccontextmanager; @asynccontextmanager async def lifespan(app): app.state.model = joblib.load(...); yield`.

**Задача 20 (`/predict`: первый инференс).** Создайте `POST /predict`: принимает `PredictRequest`, конвертирует в `pandas.DataFrame`, вызывает `model.predict()`, возвращает `PredictResponse`. Ожидаемый вывод: `{"prediction": 1, "probability": 0.78, "model_version": "v1.0"}`. В комментариях поясните: почему `DataFrame` (sklearn ожидает 2D); как названия колонок в `DataFrame` должны совпадать с train (иначе — ошибка, задача 21); почему `predict` возвращает `numpy.array` (конвертировать в `int`). Подсказка: `import pandas as pd; df = pd.DataFrame([req.model_dump()]); pred = model.predict(df)[0]`.

**Задача 21 (согласование колонок).** Убедитесь, что порядок и названия колонок в `DataFrame` совпадают с train. Продемонстрируйте ошибку: если перепутать `days_held` и `genre`, модель даст «мусор». Ожидаемый вывод: предсказание меняется, ошибки нет — **это опаснее, чем падение**. В комментариях поясните: почему sklearn **не** проверяет названия колонок (историческая проблема); как защититься (`Pipeline` + `ColumnTransformer`, задача 23); почему это самая частая ошибка в production ML. Подсказка: `df = df[["days_held", "genre", "reader_age", "book_price"]]` — явный порядок.

**Задача 22 (`handle_unknown`: новые категории).** Продемонстрируйте: модель обучена на 4 жанрах, приходит `genre="комикс"`. Что произойдёт? Ожидаемый вывод: если в модели `OneHotEncoder(handle_unknown="ignore")` — предсказание (возможно, неточное); если `handle_unknown="error"` — `ValueError` → 500. В комментариях поясните: почему новые категории — норма в production; как вернуть 422 вместо 500 (`try/except` → `HTTPException`); как настроить мониторинг неизвестных категорий (лог + алерт). Подсказка: `try: model.predict(df) except Exception as e: raise HTTPException(422, detail=str(e))`.

**Задача 23 (`Pipeline` внутри API).** Сохраните полный `Pipeline` (с `ColumnTransformer`) через `joblib`. Загрузите в API. Ожидаемый вывод: API принимает «сырые» данные (строки, категории), а `Pipeline` делает препроцессинг. В комментариях поясните: почему `Pipeline` — **единственный** правильный способ (препроцессинг вшит в модель); почему не надо вручную делать `StandardScaler` в API (data leakage, ошибки); как `Pipeline` гарантирует консистентность train/inference. Подсказка: `model = joblib.load("pipeline.joblib")` — внутри уже всё есть.

**Задача 24 (`predict_proba` vs `predict`).** Верните оба: `prediction` (класс) и `probability` (вероятность класса 1). Ожидаемый вывод: `{"prediction": 1, "probability": 0.82}`. В комментариях поясните: зачем `predict_proba` (бизнесу нужна уверенность, не только класс); как threshold влияет (`probability > 0.5` vs `> 0.3`); почему `predict_proba[:, 1]` (вторая колонка — вероятность положительного класса). Подсказка: `proba = model.predict_proba(df)[0, 1]`.

**Задача 25 (batch-инференс).** Реализуйте `POST /predict/batch`: принимает массив до 100 записей, возвращает массив предсказаний. Ожидаемый вывод: 100 предсказаний за ~10 мс (против 100 × 5 мс по одному). В комментариях поясните: почему батчинг быстрее (векторизация в numpy); как ограничить длину (`max_length=100`); почему **один** запрос лучше 100 (сеть — узкое место). Подсказка: `df = pd.DataFrame([item.model_dump() for item in batch.items])`.

**Задача 26 (логирование предсказаний).** Логируйте каждый запрос: `input`, `prediction`, `probability`, `latency_ms`. Ожидаемый вывод: в консоли — структурированный лог. В комментариях поясните: зачем логировать (debug, мониторинг дрейфа данных, аудит); почему `print` — плохо, `logging` — правильно; как добавить `request_id` (uuid) для трейсинга; что такое «data drift» (распределение входов меняется). Подсказка: `import logging; logger.info(f"predict: {req.model_dump()}, proba={proba:.3f}")`.

**Задача 27 (`model_version` в ответе).** Добавьте `model_version` в `PredictResponse` (например, из `config.py` или `model.metadata`). Ожидаемый вывод: `{"prediction": 1, "probability": 0.82, "model_version": "v1.0.0"}`. В комментариях поясните: зачем версия модели в ответе (отладка «почему вчера было иначе»); как версионировать (`git tag`, `MLflow`); почему нельзя просто «заменить model.joblib» без версии. Подсказка: `model_version: str = settings.MODEL_VERSION`.

**Задача 28 (`/health` + `/ready`).** Создайте `GET /health` (сервер жив) и `GET /ready` (модель загружена, БД доступна). Ожидаемый вывод: `/health` → 200 всегда; `/ready` → 200 только если модель в памяти. В комментариях поясните: зачем два эндпоинта (Kubernetes использует `liveness` и `readiness`); почему `/ready` может быть 503 (модель не загрузилась); как это связано с `HEALTHCHECK` в Docker (из прошлого курса). Подсказка: `@app.get("/ready") def ready(): if not app.state.model: raise HTTPException(503, "model not loaded")`.

---

## 🔴 Уровень 4. Ошибки и Dependency Injection (29–38)

**Что изучается:** `HTTPException`, кастомные exception handlers, `Depends`, `yield`-зависимости, классовые зависимости, кэширование зависимостей, `BackgroundTasks`.

**Задача 29 (`HTTPException`).** В `GET /books/{id}`: если книга не найдена — `raise HTTPException(status_code=404, detail="Book not found")`. Ожидаемый вывод: 404 с JSON `{"detail": "Book not found"}`. В комментариях поясните: почему `HTTPException` лучше `return {"error": ...}` (правильный status code); какие коды использовать (400 — плохой запрос, 404 — не найдено, 422 — валидация, 500 — сервер); как `detail` попадает в JSON. Подсказка: `from fastapi import HTTPException; raise HTTPException(404, "Book not found")`.

**Задача 30 (кастомный exception handler).** Создайте своё исключение `ModelNotReadyError` и handler через `@app.exception_handler()`, возвращающий 503. Ожидаемый вывод: при выбросе `ModelNotReadyError` — 503 с понятным JSON. В комментариях поясните: зачем кастомные исключения (чистая логика, единый формат ошибок); как `exception_handler` перехватывает; почему не `try/except` в каждом эндпоинте (DRY). Подсказка: `@app.exception_handler(ModelNotReadyError) async def handler(request, exc): return JSONResponse(503, {"detail": str(exc)})`.

**Задача 31 (`Depends`: база).** Создайте зависимость `get_model()`, которая возвращает модель из `app.state`. Используйте в `/predict`: `def predict(req: PredictRequest, model = Depends(get_model))`. Ожидаемый вывод: модель «прокидывается» в эндпоинт без глобалов. В комментариях поясните: что такое `Depends` (DI — Dependency Injection); зачем (тестируемость: можно подменить модель в тесте, см. задачу 61); почему это лучше `import model` (глобал). Подсказка: `from fastapi import Depends; def get_model(): return app.state.model`.

**Задача 32 (`yield`-зависимость).** Создайте `get_db()` через `yield`: открывает соединение, `yield`, закрывает. Ожидаемый вывод: соединение закрывается даже при ошибке. В комментариях поясните: зачем `yield` (аналог `with` — teardown после запроса); как FastAPI гарантирует закрытие; когда это критично (БД, файлы, локи). Подсказка: `def get_db(): db = Session(); try: yield db finally: db.close()`.

**Задача 33 (класс-зависимость).** Создайте `class Pagination` с `skip: int = 0`, `limit: int = 10`, и используйте `pagination: Pagination = Depends()`. Ожидаемый вывод: параметры читаются из query автоматически. В комментариях поясните: зачем классовая зависимость (группировка параметров); как FastAPI читает поля класса из query; почему это чище, чем 5 параметров в сигнатуре. Подсказка: `class Pagination: def __init__(self, skip: int = 0, limit: int = 10): ...`.

**Задача 34 (`Depends` с параметрами).** Создайте `get_current_user(api_key: str = Header(...))` — зависимость для авторизации (см. задачу 71). Ожидаемый вывод: без `X-API-Key` — 422; с ключом — доступ. В комментариях поясните: зачем зависимость с параметрами (переиспользование); как `Header` читает заголовки; почему `...` в `Header` (обязательный). Подсказка: `from fastapi import Header; def get_user(api_key: str = Header(...)): ...`.

**Задача 35 (`use_cache=False`).** Создайте зависимость, которая **каждый раз** возвращает новое значение (`use_cache=False`). Ожидаемый вывод: при вызове в двух местах — два разных объекта. В комментариях поясните: почему по умолчанию `Depends` кэширует (один вызов на запрос); когда кэш **вреден** (зависимость с побочным эффектом, разные данные); как `use_cache=False` отключает. Подсказка: `Depends(get_something, use_cache=False)`.

**Задача 36 (`BackgroundTasks`: база).** Логируйте предсказание в БД через `BackgroundTasks` — ответ отправляется **сразу**, запись в БД — в фоне. Ожидаемый вывод: latency падает (не ждём БД). В комментариях поясните: что такое `BackgroundTasks` (выполняется после отправки ответа); когда использовать (логи, email, тяжёлые записи); почему **не** для критичных операций (если упадёт — никто не узнает). Подсказка: `from fastapi import BackgroundTasks; def predict(req, bg: BackgroundTasks): bg.add_task(log_prediction, req, pred)`.

**Задача 37 (`BackgroundTasks` vs Celery).** Обсудите разницу: `BackgroundTasks` — in-process, Celery — отдельный брокер (Redis/RabbitMQ). Ожидаемый вывод: текстовое сравнение. В комментариях поясните: когда хватает `BackgroundTasks` (простые задачи, один сервер); когда нужен Celery (долгие задачи, retry, несколько воркеров); почему Celery не входит в FastAPI (это отдельный сервис). Подсказка: `pip install celery` — но для API достаточно `BackgroundTasks`.

**Задача 38 (глобальный обработчик 500).** Добавьте `@app.exception_handler(Exception)`, который логирует ошибку и возвращает 500 с `{"detail": "internal error", "request_id": "..."}`. Ожидаемый вывод: никаких «голых» трейсбеков в ответе (безопасность). В комментариях поясните: почему **нельзя** отдавать traceback клиенту (утечка путей, кода); зачем `request_id` (найти в логах); как логировать (`logger.exception`). Подсказка: `@app.exception_handler(Exception) async def handler(request, exc): logger.exception(exc); return JSONResponse(500, {...})`.

---

## 🔵 Уровень 5. Интеграция с БД (SQLAlchemy) (39–48)

**Что изучается:** `SQLAlchemy` 2.0 (Core + ORM), `Session`, модели `Base`, CRUD, `Depends(get_db)`, пагинация, связи, миграции (`alembic` — обзорно).

**Задача 39 (`SQLAlchemy`: engine + session).** Создайте `engine = create_engine("sqlite:///library.db")` и `SessionLocal = sessionmaker(bind=engine)`. Подключите в `get_db()` через `yield`. Ожидаемый вывод: `get_db()` работает как зависимость. В комментариях поясните: что такое `engine` (соединение с БД); зачем `sessionmaker` (фабрика сессий); почему SQLAlchemy 2.0 (`select()` вместо `session.query()`, см. задачу 41). Подсказка: `from sqlalchemy import create_engine; from sqlalchemy.orm import sessionmaker`.

**Задача 40 (модели `Base`).** Создайте `PredictionLog(Base)` с полями `id`, `input_data` (JSON), `prediction`, `probability`, `created_at`. Ожидаемый вывод: таблица создаётся (`Base.metadata.create_all(engine)`). В комментариях поясните: что такое ORM-модель (Python-класс ↔ таблица); зачем `Base` (реестр таблиц); как `Mapped`/`mapped_column` (SQLAlchemy 2.0) делают код типобезопасным. Подсказка: `from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column; class Base(DeclarativeBase): pass`.

**Задача 41 (CRUD: создание).** При `/predict` сохраняйте лог через `db.add(PredictionLog(...)); db.commit()`. Ожидаемый вывод: запись в БД. В комментариях поясните: зачем `add` + `commit` (add — в сессию, commit — в БД); почему **нельзя** забывать `commit` (данные потеряются); как `db.refresh(obj)` читает `id` после commit. Подсказка: `db.add(log); db.commit(); db.refresh(log)`.

**Задача 42 (CRUD: чтение + пагинация).** Создайте `GET /predictions?skip=0&limit=20` — возвращает логи с пагинацией. Ожидаемый вывод: `[{"id": 1, ...}, ...]`. В комментариях поясните: зачем пагинация (не отдавать миллион строк); как `offset`/`limit` в SQLAlchemy 2.0 (`select(...).offset(skip).limit(limit)`); почему `skip`/`limit` — стандарт REST. Подсказка: `stmt = select(PredictionLog).offset(skip).limit(limit); db.scalars(stmt).all()`.

**Задача 43 (CRUD: чтение одного + 404).** `GET /predictions/{id}`: если нет — 404. Ожидаемый вывод: 404 для несуществующего id. В комментариях поясните: как `db.get(PredictionLog, id)` работает; почему 404, а не пустой объект; как `response_model` фильтрует. Подсказка: `obj = db.get(PredictionLog, id); if not obj: raise HTTPException(404)`.

**Задача 44 (CRUD: update).** `PATCH /predictions/{id}` — обновляет `probability` (например, ручная коррекция). Ожидаемый вывод: обновлённая запись. В комментариях поясните: чем `PUT` отличается от `PATCH` (полная замена vs частичное обновление); как `model_dump(exclude_unset=True)` в Pydantic помогает (обновлять только переданные поля); почему `commit` обязателен. Подсказка: `for k, v in data.model_dump(exclude_unset=True).items(): setattr(obj, k, v)`.

**Задача 45 (CRUD: delete).** `DELETE /predictions/{id}` — удаляет, возвращает 204. Ожидаемый вывод: 204 No Content. В комментариях поясните: почему 204 (нет тела); как `db.delete(obj)`; почему в реальной аналитике логи **не** удаляют (аудит), а помечают `is_deleted`. Подсказка: `db.delete(obj); db.commit(); return Response(status_code=204)`.

**Задача 46 (связи: Reader ↔ Prediction).** Добавьте `Reader(Base)` и связь `PredictionLog.reader_id` (FK). Реализуйте `GET /readers/{id}/predictions` через `relationship`. Ожидаемый вывод: все предсказания читателя. В комментариях поясните: как `relationship` работает в ORM (ленивая загрузка); что такое N+1 проблема (и как `selectinload` спасает); почему FK + relationship — основа реляционных БД. Подсказка: `reader_id: Mapped[int] = mapped_column(ForeignKey("readers.id")); reader: Mapped["Reader"] = relationship(back_populates="predictions")`.

**Задача 47 (`alembic`: миграции — обзорно).** Опишите, зачем миграции (изменение схемы без потери данных) и как выглядит `alembic revision --autogenerate`. Ожидаемый вывод: текстовое объяснение + пример. В комментариях поясните: почему `create_all()` **не** подходит для production (не умеет менять существующие таблицы); как `alembic upgrade head`; почему миграции — обязательны для командной работы. Подсказка: `pip install alembic; alembic init migrations`.

**Задача 48 (транзакции).** Оберните «создать читателя + лог» в одну транзакцию: при ошибке — `rollback`. Ожидаемый вывод: если лог упал — читатель тоже не создался. В комментариях поясните: что такое ACID; зачем `db.begin()`; почему `try/except → rollback` обязателен; как SQLAlchemy автоматически коммитит в конце сессии. Подсказка: `with db.begin(): db.add(reader); db.add(log)`.

---

## 🟣 Уровень 6. Асинхронность (49–56)

**Что изучается:** `async def` vs `def`, `await`, `asyncio`, `httpx` (async), `asyncio.gather`, блокирующий код в async, `run_in_threadpool`.

**Задача 49 (`async def`: база).** Перепишите `/predict` как `async def predict(...)`. Ожидаемый вывод: работает так же, но **не** блокирует event loop. В комментариях поясните: чем `async def` отличается от `def` (event loop vs thread pool); почему FastAPI запускает `def` в thread pool (безопасно, но медленнее); когда `async` обязателен (I/O: БД, HTTP). Подсказка: `@app.post("/predict") async def predict(...): ...`.

**Задача 50 (`await httpx`).** Создайте эндпоинт, который делает внешний HTTP-запрос через `httpx.AsyncClient` (например, «курс валют»). Ожидаемый вывод: запрос не блокирует сервер. В комментариях поясните: почему `requests` **нельзя** в `async def` (блокирует event loop); чем `httpx` лучше (async + sync); как `async with httpx.AsyncClient()` управляет соединением. Подсказка: `import httpx; async with httpx.AsyncClient() as client: r = await client.get(...)`.

**Задача 51 (`asyncio.gather`).** Сделайте 3 параллельных запроса к внешнему API через `asyncio.gather`. Ожидаемый вывод: 3 запроса за ~время одного (параллельно). В комментариях поясните: что делает `gather` (запускает корутины параллельно); почему это быстрее последовательных `await`; когда `gather` **не** помогает (CPU-bound). Подсказка: `results = await asyncio.gather(fetch(1), fetch(2), fetch(3))`.

**Задача 52 (блокирующий код в async).** Продемонстрируйте: `time.sleep(2)` в `async def` **блокирует** весь сервер. Исправьте через `asyncio.sleep(2)` или `run_in_threadpool`. Ожидаемый вывод: с `time.sleep` второй запрос ждёт; с `asyncio.sleep` — параллельно. В комментариях поясните: почему `time.sleep` — «убийца» async; когда `run_in_threadpool` (`from starlette.concurrency import run_in_threadpool`); почему ML-инференс (CPU-bound) лучше делать в `def`, а не `async def`. Подсказка: `await asyncio.sleep(2)`.

**Задача 53 (`async` + SQLAlchemy).** Перепишите `get_db()` на async: `create_async_engine`, `AsyncSession`. Ожидаемый вывод: БД-запросы не блокируют. В комментариях поясните: почему async SQLAlchemy требует `aiosqlite`/`asyncpg`; как `async with AsyncSession() as session: await session.execute(...)`; когда async БД оправдан (много I/O). Подсказка: `pip install aiosqlite; from sqlalchemy.ext.asyncio import create_async_engine, AsyncSession`.

**Задача 54 (`async` vs `def` для ML).** Обсудите: ML-инференс (CPU-bound) **блокирует** поток. Что лучше: `def` (thread pool) или `async def` + `run_in_threadpool`? Ожидаемый вывод: для sklearn — `def` (FastAPI сам запустит в thread pool); для GPU/PyTorch — отдельная история. В комментариях поясните: почему `async def` + синхронный sklearn — **худший** вариант; как `anyio.to_thread.run_sync`; почему event loop — «однопоточный». Подсказка: `def predict(...)` — FastAPI сам сделает thread pool.

**Задача 55 (`asyncio.timeout`).** Оберните внешний запрос в таймаут: `async with asyncio.timeout(5): ...`. Ожидаемый вывод: при превышении — `TimeoutError` → 504. В комментариях поясните: зачем таймауты (не зависать на «мёртвом» сервисе); как обработать (`try/except TimeoutError → HTTPException(504)`); почему `httpx` тоже имеет `timeout` (используйте оба). Подсказка: `async with asyncio.timeout(5): await client.get(...)`.

**Задача 56 (семафор: ограничение параллелизма).** Ограничьте число одновременных запросов к внешнему API через `asyncio.Semaphore(10)`. Ожидаемый вывод: не более 10 параллельных запросов. В комментариях поясните: зачем семафор (защита внешнего API от «флуда»); как `async with sem:`; когда `gather` без семафора «положит» внешний сервис. Подсказка: `sem = asyncio.Semaphore(10); async with sem: ...`.

---

## 🔴 Уровень 7. Тестирование (57–66)

**Что изучается:** `TestClient`, `pytest`, фикстуры (`client`, `db`), `mock` модели, тестирование ошибок, `httpx.AsyncClient` для async, покрытие.

**Задача 57 (`TestClient`: первый тест).** Установите `httpx` и создайте `test_main.py` с `TestClient(app)`. Тест: `client.get("/")` → 200. Ожидаемый вывод: `1 passed`. В комментариях поясните: что такое `TestClient` (обёртка над `httpx`, не поднимает реальный сервер); зачем (быстрые тесты без `uvicorn`); почему `TestClient` синхронный, а FastAPI — async (внутри запускает event loop). Подсказка: `from fastapi.testclient import TestClient; client = TestClient(app)`.

**Задача 58 (фикстура `client`).** Создайте `tests/conftest.py` с `@pytest.fixture def client(): return TestClient(app)`. Используйте во всех тестах. Ожидаемый вывод: без дублирования `TestClient(app)` в каждом тесте. В комментариях поясните: зачем фикстуры в API-тестах (единый клиент); почему `scope="session"` может быть опасен (состояние между тестами); как `yield` для teardown. Подсказка: `@pytest.fixture def client(): with TestClient(app) as c: yield c`.

**Задача 59 (тест POST).** Напишите тест: `POST /predict` с валидным JSON → 200 + `prediction in [0, 1]`. Ожидаемый вывод: `1 passed`. В комментариях поясните: почему тестировать **контракт** (JSON-схему), а не точное значение модели (модель может измениться); как `response.json()`; почему `assert "prediction" in data`. Подсказка: `r = client.post("/predict", json={"days_held": 14, "genre": "роман"}); assert r.status_code == 200`.

**Задача 60 (тест ошибок).** Тест: `POST /predict` с `days_held=-5` → 422; с `genre="комикс"` → 422. Ожидаемый вывод: 2 теста, оба passed. В комментариях поясните: почему тестировать **валидацию** важнее, чем happy path; как проверить `detail` (`r.json()["detail"]`); почему 422 — стандарт FastAPI. Подсказка: `assert r.status_code == 422`.

**Задача 61 (`mock` модели).** Подмените модель в `app.state.model` через `mocker` или `monkeypatch` на фейковую (возвращает фиксированное значение). Ожидаемый вывод: тест не зависит от реальной модели. В комментариях поясните: зачем мокать (тесты быстрые, детерминированные); как `monkeypatch.setattr(app.state, "model", FakeModel())`; почему **не** мокать Pydantic/валидацию (это часть контракта). Подсказка: `class FakeModel: def predict(self, X): return [1]; def predict_proba(self, X): return [[0.2, 0.8]]`.

**Задача 62 (фикстура `db` в тестах).** Создайте тестовую БД (`sqlite:///:memory:`) и фикстуру `db` с откатом после каждого теста. Ожидаемый вывод: тесты не «загрязняют» реальную БД. В комментариях поясните: почему `:memory:` (быстро, изолированно); как `Base.metadata.create_all(engine)` в фикстуре; почему **нельзя** тестировать на production БД. Подсказка: `engine = create_engine("sqlite:///:memory:", connect_args={"check_same_thread": False})`.

**Задача 63 (override dependencies).** Используйте `app.dependency_overrides[get_db] = override_get_db` в тестах. Ожидаемый вывод: эндпоинт использует тестовую БД. В комментариях поясните: что такое `dependency_overrides` (подмена DI в тестах); зачем (не менять код ради тестов); как очистить после теста (`app.dependency_overrides.clear()`). Подсказка: `app.dependency_overrides[get_db] = lambda: test_db`.

**Задача 64 (тест async через `httpx.AsyncClient`).** Для async-эндпоинтов используйте `httpx.AsyncClient(app=app, base_url="http://test")` в `pytest.mark.asyncio`. Ожидаемый вывод: async-тест работает. В комментариях поясните: почему `TestClient` не всегда подходит (async); как `pytest-asyncio` (`@pytest.mark.asyncio`); чем `AsyncClient` лучше (реальная async-семантика). Подсказка: `pip install pytest-asyncio; @pytest.mark.asyncio async def test_x(): async with AsyncClient(app=app, base_url="http://test") as ac: ...`.

**Задача 65 (coverage для API).** Запустите `pytest --cov=app`. Ожидаемый вывод: покрытие эндпоинтов, схем, утилит. В комментариях поясните: почему 100% — не цель (не все ветки ошибок); как `# pragma: no cover`; где покрытие критично (эндпоинты, валидация). Подсказка: `pytest --cov=app --cov-report=term-missing`.

**Задача 66 (тест `health` / `ready`).** Тест: `/health` → 200 всегда; `/ready` → 503 если модель не загружена. Ожидаемый вывод: тесты для health/ready. В комментариях поясните: почему health/ready — критичны для Kubernetes (задача 28); как сэмулировать «модель не загружена» (`app.state.model = None`); почему 503, а не 500. Подсказка: `monkeypatch.setattr(app.state, "model", None); assert client.get("/ready").status_code == 503`.

---

## 🟠 Уровень 8. Безопасность и CORS (67–74)

**Что изучается:** API keys (`Header`), OAuth2 (обзорно), `HTTPBearer`, CORS (`CORSMiddleware`), rate limiting (обзорно), HTTPS, secrets.

**Задача 67 (API Key: `Header`).** Создайте зависимость `verify_api_key(x_api_key: str = Header(...))`, которая проверяет ключ из `.env`. Ожидаемый вывод: без ключа — 401/422; с ключом — доступ. В комментариях поясните: зачем API key (простая авторизация для сервисов); почему ключ в **заголовке**, а не в URL (URL логируется); как `secrets.compare_digest` (защита от timing-атак). Подсказка: `from fastapi import Header, HTTPException; def verify(x_api_key: str = Header(...)): if x_api_key != settings.API_KEY: raise HTTPException(401)`.

**Задача 68 (`Depends` для защиты группы).** Примените `verify_api_key` ко **всему** роутеру через `APIRouter(dependencies=[Depends(verify_api_key)])`. Ожидаемый вывод: все эндпоинты роутера защищены. В комментариях поясните: зачем защищать группу (DRY); как `dependencies` на роутере (применяется ко всем); почему `/health` **не** защищают. Подсказка: `router = APIRouter(dependencies=[Depends(verify_api_key)])`.

**Задача 69 (`HTTPBearer`).** Реализуйте Bearer-токен: `HTTPBearer()` читает `Authorization: Bearer <token>`. Ожидаемый вывод: Swagger показывает кнопку «Authorize». В комментариях поясните: чем Bearer отличается от API key (стандарт OAuth2, JWT); зачем `HTTPBearer` (`from fastapi.security import HTTPBearer`); как Swagger автоматически добавляет заголовок. Подсказка: `from fastapi.security import HTTPBearer; security = HTTPBearer()`.

**Задача 70 (JWT: обзорно).** Опишите flow JWT: login → token → `Authorization: Bearer`. Ожидаемый вывод: текстовое объяснение + пример декодирования (`python-jose`). В комментариях поясните: что такое JWT (header.payload.signature); почему **нельзя** хранить пароли в JWT; как `python-jose` (`jwt.encode`, `jwt.decode`); почему JWT — стандарт для SPA. Подсказка: `pip install python-jose[cryptography]`.

**Задача 71 (`CORSMiddleware`).** Добавьте CORS для `http://localhost:3000` (React):
```python
app.add_middleware(CORSMiddleware, allow_origins=["http://localhost:3000"], allow_methods=["*"], allow_headers=["*"])
```
Ожидаемый вывод: фронт может обращаться к API. В комментариях поясните: что такое CORS (Same-Origin Policy); почему браузер блокирует запросы с другого порта; почему `allow_origins=["*"]` — плохо для production (безопасность); что такое preflight-запрос (`OPTIONS`). Подсказка: `from fastapi.middleware.cors import CORSMiddleware`.

**Задача 72 (rate limiting: обзорно).** Опишите rate limiting (не более 100 запросов в минуту с одного IP) через `slowapi` или middleware. Ожидаемый вывод: текстовое объяснение + пример. В комментариях поясните: зачем rate limiting (защита от DDoS, справедливость); как `slowapi`; почему без лимита модель может «упасть» под нагрузкой. Подсказка: `pip install slowapi`.

**Задача 73 (secrets в `.env`).** Убедитесь: `API_KEY`, `DB_URL`, `SECRET_KEY` — в `.env`, а не в коде. `.env` в `.gitignore`. Ожидаемый вывод: `git grep "API_KEY"` не находит ключ в коде. В комментариях поясните: почему **нельзя** хардкодить секреты (утечка через GitHub); что такое `.env.example` (шаблон без секретов); как `pydantic-settings` читает `.env`; почему в production — vault (AWS Secrets Manager, HashiCorp Vault). Подсказка: `git grep -i "api_key"` — должно быть пусто.

**Задача 74 (HTTPS + заголовки безопасности).** Обсудите: HTTPS на уровне reverse proxy (nginx/traefik), заголовки `X-Content-Type-Options`, `X-Frame-Options`, `Strict-Transport-Security`. Ожидаемый вывод: текстовое объяснение + пример middleware. В комментариях поясните: почему HTTPS обязателен (пароли, токены); почему FastAPI **не** делает HTTPS сам (это задача прокси); как `TrustedHostMiddleware` (защита от Host header injection). Подсказка: `from starlette.middleware.trustedhost import TrustedHostMiddleware`.

---

## ⚫ Уровень 9. Продвинутое и деплой (75–85)

**Что изучается:** WebSockets (обзорно), StreamingResponse, `APIRouter` с версионированием (`/api/v1`), `uvicorn --workers`, Gunicorn, multi-stage Dockerfile для ML, `HEALTHCHECK`, `nginx`, логирование, мониторинг (Prometheus — обзорно).

**Задача 75 (`/api/v1` версионирование).** Вынесите все роутеры под `prefix="/api/v1"`. Ожидаемый вывод: `POST /api/v1/predict`, `GET /api/v1/predictions`. В комментариях поясните: зачем версионирование (ломающие изменения без поломки клиентов); почему `/v1` в URL (проще, чем заголовки); как `include_router(..., prefix="/api/v1")`. Подсказка: `app.include_router(predict_router, prefix="/api/v1")`.

**Задача 76 (`StreamingResponse`).** Реализуйте эндпоинт, который отдаёт CSV-файл с логами через `StreamingResponse`. Ожидаемый вывод: файл скачивается, не грузится в память целиком. В комментариях поясните: зачем streaming (большие файлы, память); как `io.StringIO` + генератор; чем `StreamingResponse` отличается от `FileResponse` (первый — генератор, второй — файл с диска). Подсказка: `from fastapi.responses import StreamingResponse`.

**Задача 77 (WebSockets: обзорно).** Создайте `@app.websocket("/ws")`, который принимает сообщение и возвращает эхо. Ожидаемый вывод: `wscat -c ws://localhost:8000/ws` работает. В комментариях поясните: чем WebSocket отличается от HTTP (двусторонний канал); когда нужен (чат, live-обновления); почему для ML-инференса обычно не нужен (HTTP достаточно). Подсказка: `@app.websocket("/ws") async def ws(websocket: WebSocket): await websocket.accept(); ...`.

**Задача 78 (multi-stage Dockerfile для ML).** Напишите `Dockerfile`: stage 1 — `pip wheel` для зависимостей; stage 2 — `python:3.11-slim`, `COPY --from=builder`, `COPY app/`, `CMD uvicorn`. Ожидаемый вывод: образ ≤ 400 МБ (вместо 2 ГБ). В комментариях поясните: почему `scikit-learn` + `pandas` тяжёлые; зачем `--no-cache-dir`; как `.dockerignore` (`.venv`, `.git`) уменьшает контекст; почему **не** `python:3.11` (полный образ ~1 ГБ). Подсказка: `FROM python:3.11-slim AS builder; ... FROM python:3.11-slim; COPY --from=builder /wheels /wheels`.

**Задача 79 (`uvicorn --workers`).** Запустите `uvicorn app.main:app --host 0.0.0.0 --port 8000 --workers 4`. Ожидаемый вывод: 4 процесса обрабатывают запросы параллельно. В комментариях поясните: зачем workers (использовать все ядра); почему `--workers 1` + `async` может быть достаточно (I/O-bound); почему **не** ставьте workers больше, чем ядер (GIL, память). Подсказка: `uvicorn app.main:app --workers 4`.

**Задача 80 (Gunicorn + Uvicorn workers).** Для production: `gunicorn app.main:app -k uvicorn.workers.UvicornWorker -w 4`. Ожидаемый вывод: Gunicorn управляет процессами, Uvicorn — event loop. В комментариях поясните: зачем Gunicorn (graceful restart, мониторинг процессов); почему `UvicornWorker`; когда хватает чистого Uvicorn (Docker + Kubernetes сами управляют). Подсказка: `pip install gunicorn; gunicorn app.main:app -k uvicorn.workers.UvicornWorker -w 4`.

**Задача 81 (`HEALTHCHECK` в Dockerfile).** Добавьте `HEALTHCHECK CMD python -c "import httpx; httpx.get('http://localhost:8000/health')" || exit 1`. Ожидаемый вывод: `docker ps` показывает `(healthy)`. В комментариях поясните: зачем (Docker/K8s знают, жив ли контейнер); как `--interval`, `--timeout`, `--retries`; почему без healthcheck K8s не перезапустит «зависший» контейнер. Подсказка: `HEALTHCHECK --interval=30s --timeout=3s --retries=3 CMD ...`.

**Задача 82 (логирование JSON).** Настройте `logging` в JSON-формате (`python-json-logger`), чтобы логи легко парсились в ELK/Loki. Ожидаемый вывод: `{"time": "...", "level": "INFO", "message": "..."}`. В комментариях поясните: зачем JSON-логи (парсинг, поиск); почему `print` — плохо; как `structlog` (альтернатива); что такое correlation id (задача 26). Подсказка: `pip install python-json-logger; logging.basicConfig(...)`.

**Задача 83 (reverse proxy: nginx).** Опишите, зачем nginx перед FastAPI (HTTPS, статика, load balancing, rate limiting). Ожидаемый вывод: текстовое объяснение + пример `nginx.conf`. В комментариях поясните: почему FastAPI **не** должен быть «лицом в интернет»; как `proxy_pass http://app:8000`; что такое `X-Forwarded-For` (реальный IP клиента). Подсказка: `location / { proxy_pass http://app:8000; }`.

**Задача 84 (мониторинг: Prometheus — обзорно).** Опишите метрики, которые нужно собирать: `request_count`, `request_latency`, `prediction_distribution`, `error_rate`. Ожидаемый вывод: текстовое объяснение + пример `prometheus-fastapi-instrumentator`. В комментариях поясните: зачем мониторинг (узнать о проблеме раньше пользователей); как Prometheus + Grafana; почему latency p95 важнее среднего. Подсказка: `pip install prometheus-fastapi-instrumentator`.

**Задача 85 (финальный проект «под ключ»).** Соберите всё: (1) `app/` с роутерами `/predict`, `/predictions`, `/readers`; (2) Pydantic-схемы с валидацией; (3) загрузка `Pipeline` через `lifespan`; (4) `SQLAlchemy` + SQLite для логов; (5) `BackgroundTasks` для логирования; (6) API key + CORS; (7) `pytest` + `TestClient` + coverage ≥ 80%; (8) multi-stage `Dockerfile` ≤ 400 МБ; (9) `README` с примерами `curl`; (10) `docker-compose.yml` (app + postgres — обзорно). Ожидаемый результат: production-ready ML-сервис. В комментариях опишите: как разбить на модули; почему тесты важны; как масштабируется (K8s); какие следующие шаги (MLflow, Prometheus, CI/CD). Подсказка: начните с `/predict`, добавляйте по одному эндпоинту, сразу пишите тест.

---
